# Sprint 4 — Modelo de predição de inadimplência e scoring de recuperabilidade

**Projeto Integrador — BIINAR, Eixo II 2026/2 | UFT Palmas**
**Sistema de Inteligência Tributária — priorização da cobrança da dívida ativa de Palmas-TO**

| | |
|---|---|
| **Equipe** | Davi Pereira (mat. ___), Laura Nicolle (mat. ___), Márcio Araújo (mat. ___), Matheus Felipe (mat. ___), Marcus Vinicius (mat. ___) |
| **Data de execução** | 15/11/2026 |
| **Repositório** | https://github.com/LIAdrien/projeto-git-uft |

## Objetivo

Treinar e avaliar um modelo de classificação de inadimplência e um modelo
complementar de scoring de recuperabilidade (faixas alta, média e baixa),
para apoiar a priorização da cobrança da dívida ativa.

Trilha deste notebook: **EDA → pré-processamento → treinamento → avaliação →
serialização do modelo em `/models`**.

## Proveniência dos dados

| Camada | Origem | Natureza |
|---|---|---|
| Arrecadação agregada | API do portal de transparência, coletada em 20/10/2026 | **Real** |
| Carteira de contribuintes | Gerador calibrado (`src/sintetico.py`) | **Sintética** |

**Declaração obrigatória.** A camada individual de dívida ativa é sintética,
calibrada pelos agregados oficiais da Sefin. O registro individual do
contribuinte é dado pessoal protegido pela LGPD e não é publicado por nenhuma
API municipal.

As métricas deste notebook demonstram que **o pipeline funciona** — não que o
modelo funcionaria no mundo real. Qualquer estimativa de desempenho real exige
recalibração sobre dados reais, mediante convênio.

## Ambiente e reprodutibilidade

Dependências em `requirements.txt`. Os números deste notebook são
reprodutíveis com a semente fixa (`SEMENTE = 42`), definida na célula de
imports da seção 1.

In [ ]:
# Preparação do ambiente. Só age no Google Colab; em máquina local
# (repositório clonado, notebook em notebooks/) não faz nada.
import shutil
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO = Path("/content/projeto-git-uft")
    POC = Path("/content/poc-referencia")      # repositório de referência da aula (carteira)

    if not REPO.exists():
        !git clone https://github.com/LIAdrien/projeto-git-uft.git {REPO}
    (REPO / "notebooks").mkdir(exist_ok=True)
    (REPO / "data").mkdir(exist_ok=True)

    carteira = REPO / "data" / "carteira_contribuintes.csv"
    if not carteira.exists():
        if not POC.exists():
            !git clone https://github.com/davidnadlerprata/poc-inteligencia-tributaria-palmas.git {POC}
        shutil.copy(POC / "data" / "carteira_contribuintes.csv", carteira)

    if not (REPO / "requirements.txt").exists() and POC.exists():
        shutil.copy(POC / "requirements.txt", REPO)

    %cd {REPO}/notebooks

print("Pasta atual:", Path.cwd())
print("Carteira   :", Path("../data/carteira_contribuintes.csv").exists())

/content/projeto-git-uft/notebooks
Pasta atual: /content/projeto-git-uft/notebooks
Carteira   : True


## 1. Preparação: importações, semente e caminhos

Importações do notebook, semente fixa (`SEMENTE = 42`), caminhos relativos a
`notebooks/` e versões das bibliotecas, para que qualquer pessoa reproduza os
mesmos números.

In [ ]:
# Seção 1 - Código


import json
import platform
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEMENTE = 42
np.random.seed(SEMENTE)

# O notebook mora em notebooks/; dados e modelos ficam um nível acima.
DADOS = Path("../data")
MODELOS = Path("../models")
MODELOS.mkdir(exist_ok=True)

print(f"python  {platform.python_version()}")
print(f"pandas  {pd.__version__}")
print(f"sklearn {sklearn.__version__}")
print(f"joblib  {joblib.__version__}")

python  3.13.18
pandas  2.2.3
sklearn 1.6.1
joblib  1.6.0


## 2. Carga, exclusão do grupo protegido e EDA

Esta seção carrega a carteira, **exclui os beneficiários do IPTU Social antes
de qualquer outra etapa** e faz uma EDA curta, voltada às decisões de modelagem.


In [ ]:
dados = pd.read_csv(DADOS / "carteira_contribuintes.csv", sep=";")
print(f"{len(dados)} registros, {dados.shape[1]} colunas")

# Verificação de integridade antes de excluir
colunas_necessarias = [
    "iptu_social",
    "inadimplente_proximo_exercicio",
    "valor_divida_consolidada",
    "divida_recuperada_12m",
]
faltando = [c for c in colunas_necessarias if c not in dados.columns]
assert not faltando, f"colunas ausentes: {faltando}"
assert dados["iptu_social"].notna().all(), "iptu_social com valores vazios"

# Beneficiários do IPTU Social saem ANTES de qualquer split, EDA ou treino.
universo = dados[dados["iptu_social"] == 0].copy()
print(f"excluídos pelo IPTU Social: {len(dados) - len(universo)} | universo: {len(universo)}")

assert universo["iptu_social"].sum() == 0, "grupo protegido ainda presente!"

20000 registros, 17 colunas
excluídos pelo IPTU Social: 1393 | universo: 18607


**Por que a exclusão vem primeiro.** Idosos, aposentados, pensionistas e
pessoas com deficiência de baixa renda são isentos e não podem ser alvo de
ação preditiva de cobrança. Se a exclusão vier depois do split, eles já estarão
dentro do conjunto de treino, e nenhuma mensagem de erro vai avisar.
Daqui em diante, toda a trilha parte de `universo`, nunca de `dados`.

### EDA orientada a decisão

A análise exploratória completa foi feita na Sprint de coleta e ETL. Aqui
mostramos apenas os fatos que justificam as decisões de modelagem seguintes.

In [ ]:
# 1) Balanceamento do alvo do Modelo 1 -- decide se precisa de class_weight ou reamostragem
print("Modelo 1 -- inadimplente_proximo_exercicio")
print(universo["inadimplente_proximo_exercicio"].value_counts(normalize=True).round(3))

# 2) Universo do Modelo 2 -- só faz sentido para quem tem dívida inscrita
tem_divida = universo["valor_divida_consolidada"] > 0
print(f"\ncom dívida inscrita: {tem_divida.sum()} de {len(universo)}")
print("Modelo 2 -- divida_recuperada_12m")
print(universo.loc[tem_divida, "divida_recuperada_12m"].value_counts(normalize=True).round(3))

Modelo 1 -- inadimplente_proximo_exercicio
inadimplente_proximo_exercicio
1    0.507
0    0.493
Name: proportion, dtype: float64

com dívida inscrita: 15361 de 18607
Modelo 2 -- divida_recuperada_12m
divida_recuperada_12m
1    0.516
0    0.484
Name: proportion, dtype: float64


### Decisões que a EDA sustenta

1. Os dois alvos estão equilibrados (50,7% de inadimplentes no Modelo 1 e
   51,6% de recuperadas no Modelo 2) → não usaremos reamostragem nem
   `class_weight`.
2. O Modelo 2 usa só os 15.361 registros com dívida inscrita.
3. Os 1.393 beneficiários do IPTU Social saíram antes de qualquer treino;
   o universo de trabalho tem 18.607 registros.


## 3. Split temporal

Esta é a seção central da metodologia (E1): **nada do futuro pode informar o
passado no treino**.

### 3.1 Por que não usamos `train_test_split` aleatório

Um split aleatório mistura períodos: o modelo treinaria com registros recentes
e seria testado em registros antigos, algo impossível na operação real, em que
só o passado está disponível para prever o futuro. Com dados reais, isso
superestima o desempenho. Por isso o corte é feito por data.

### 3.2 Por que a data não é derivada de `tempo_inadimplencia_dias`

A carteira não tem coluna de data. Derivá-la de `tempo_inadimplencia_dias`
faria o corte temporal separar exatamente pelo preditor mais forte do Modelo 2:
o teste ficaria só com dívidas recentes, de recuperação muito diferente da do
treino. O eixo temporal precisa ser **independente dos atributos preditivos**;
caso contrário, é a própria feature disfarçada.

### 3.3 Eixo temporal: competência do registro

Em dados reais, a dívida ativa é uma foto mensal, e o mês de competência define
o eixo temporal. Como a carteira sintética não tem esse campo, **criamos** uma
competência por sorteio, independente de qualquer atributo (jan/2024 a
out/2026).

**Limitação declarada.** Como a competência é sorteada, não há deriva temporal
nesta base: passado e futuro têm a mesma distribuição, e o split temporal
equivale, por construção, a um split aleatório. O experimento exercita a
mecânica correta (cortar por data, validar no futuro, provar a ausência de
vazamento), mas não mostra o efeito que ela protege. Com dados reais, em que a
economia e a política de cobrança mudam de um ano para outro, a diferença
apareceria.

In [ ]:
rng = np.random.default_rng(SEMENTE)

# Competência: mês de referência do registro. Independente de qualquer atributo.
meses = pd.date_range("2024-01-01", "2026-10-01", freq="MS")
universo["competencia"] = rng.choice(meses, size=len(universo))

# Ordem cronológica: a validação temporal da seção 3.5 depende dela.
universo = universo.sort_values("competencia").reset_index(drop=True)

In [ ]:
CORTE_TESTE = pd.Timestamp("2026-04-01")      # os últimos 7 meses ficam para teste

treino = universo[universo["competencia"] < CORTE_TESTE]
teste = universo[universo["competencia"] >= CORTE_TESTE]

print(f"treino: {len(treino):5d} ({len(treino) / len(universo):.0%})"
      f" | até {treino['competencia'].max().date()}")
print(f"teste:  {len(teste):5d} ({len(teste) / len(universo):.0%})"
      f" | de {teste['competencia'].min().date()}")

treino: 14877 (80%) | até 2026-03-01
teste:   3730 (20%) | de 2026-04-01


### 3.4 Provando que não há vazamento

Três verificações em código: sem sobreposição temporal, sem registro nos dois
lados e distribuições comparáveis entre treino e teste.

In [ ]:
colunas = ["valor_venal", "exercicios_inadimplentes_5a", "inadimplente_proximo_exercicio"]
ausentes = [c for c in colunas + ["id_contribuinte"] if c not in universo.columns]
assert not ausentes, f"colunas ausentes na carteira: {ausentes}"

# 1) Não há sobreposição temporal
assert treino["competencia"].max() < teste["competencia"].min(), "sobreposição temporal!"

# 2) Nenhum registro está nos dois lados
#    (nesta carteira cada contribuinte aparece uma única vez)
repetidos = set(treino["id_contribuinte"]) & set(teste["id_contribuinte"])
assert not repetidos, f"{len(repetidos)} contribuintes em treino e teste"

# 3) As distribuições são comparáveis (se não forem, o corte está ruim)
comparacao_split = pd.DataFrame({"treino": treino[colunas].mean(),
                                 "teste": teste[colunas].mean()}).round(3)
print(comparacao_split)

                                    treino       teste
valor_venal                     293925.093  291649.121
exercicios_inadimplentes_5a          1.651       1.646
inadimplente_proximo_exercicio       0.508       0.503


### 3.5 Validação temporal

A validação sai de dentro do treino, respeitando a ordem. O teste fica
guardado até a seção 7 e é usado uma única vez. Comparação de modelos,
tunagem e escolha do limiar usam estas dobras.

Regras: nunca usar `KFold` ou `cross_val_score` com embaralhamento. O
`TimeSeriesSplit` divide por **posição** na tabela, e só é temporal porque
`universo` foi ordenado por competência na seção 3.3; não reordene `universo`
depois.

In [ ]:
# 4 dobras: cada uma ajusta no passado e valida no futuro imediato
validacao_temporal = TimeSeriesSplit(n_splits=4)

# Seguro contra reordenação silenciosa: sem ordem cronológica, as dobras vazam.
assert treino["competencia"].is_monotonic_increasing, "treino fora de ordem cronológica!"

### 3.6 Síntese da seção 3

- **Corte explícito:** o treino vai de jan/2024 a mar/2026 e o teste de
  abr/2026 a out/2026 (14.877 e 3.730 registros, 80% e 20%).
- **Sem vazamento temporal:** a última competência do treino é anterior à
  primeira do teste, e nenhum contribuinte aparece nos dois lados.
- **Distribuições comparáveis:** valor venal, exercícios inadimplentes e taxa de
  inadimplência têm médias muito próximas entre treino e teste, o que indica
  que o corte não separou perfis diferentes.
- **Limitação:** a competência foi sorteada, então este split exercita a
  metodologia correta, mas não mede o efeito de uma deriva temporal real.

## 4. Pré-processamento dentro do pipeline
### O erro que mais reprova

Ajustar o `StandardScaler` (ou qualquer transformação) sobre treino e teste
juntos faz informação do teste vazar para o treino: a média e o desvio passam a
ser calculados também com o teste, e a métrica final fica otimista, sem erro
nem aviso. Por isso todo o pré-processamento fica dentro de um `Pipeline`, que
só ajusta (`fit`) o que recebe e refaz o ajuste a cada dobra de validação,
usando apenas a parte de ajuste.

In [ ]:
CATEGORICOS = ["tipo_imovel", "setor_urbano"]
NUMERICOS = ["area_construida_m2", "valor_venal", "valor_iptu_anual",
             "distancia_centro_km", "exercicios_inadimplentes_5a",
             "tempo_inadimplencia_dias", "valor_divida_consolidada",
             "qtd_parcelamentos_anteriores", "parcelamento_rompido",
             "qtd_notificacoes"]
ATRIBUTOS = CATEGORICOS + NUMERICOS

ALVO = "inadimplente_proximo_exercicio"          # Modelo 1

ausentes = [c for c in ATRIBUTOS + [ALVO] if c not in treino.columns]
assert not ausentes, f"colunas ausentes na carteira: {ausentes}"

X_treino, y_treino = treino[ATRIBUTOS], treino[ALVO]
X_teste, y_teste = teste[ATRIBUTOS], teste[ALVO]

# Três coisas que NÃO podem estar entre os atributos:
PROIBIDOS = {"inadimplente_proximo_exercicio", "divida_recuperada_12m",   # os alvos
             "faixa_renda_setor",                                        # grupo protegido
             "iptu_social", "competencia", "id_contribuinte"}
assert not (PROIBIDOS & set(ATRIBUTOS)), "coluna proibida entre os atributos!"

print(f"{len(ATRIBUTOS)} atributos | X_treino {X_treino.shape} | X_teste {X_teste.shape}")

12 atributos | X_treino (14877, 12) | X_teste (3730, 12)


### Por que cada coluna fica de fora

- **Os alvos:** alvo entre os preditores é o vazamento mais grosseiro; a métrica
  iria para perto de 1,0 e o modelo não serviria para nada.
- **`faixa_renda_setor`:** é o grupo protegido. Usá-la como preditor seria
  discriminação direta por renda; ela fica guardada para a auditoria de
  fairness da Sprint 5.
- **`iptu_social`:** dado sensível (LGPD, Art. 5º, II), usado só para excluir.
- **`competencia` e `id_contribuinte`:** organizam o experimento, não preveem.

### A forma correta: tudo dentro de um `Pipeline`

O pré-processamento imputa os numéricos pela mediana e os padroniza, e codifica
os categóricos com *one-hot*. O `handle_unknown="ignore"` é necessário no split
temporal: o teste pode trazer uma categoria que não existia no treino (um setor
novo), e sem ele a avaliação quebraria.

In [ ]:
def montar_preparo():
    """Pré-processamento. Uma instância nova a cada chamada, para cada modelo."""
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICOS),
        ("num", Pipeline([
            ("imputacao", SimpleImputer(strategy="median")),
            ("escala", StandardScaler()),
        ]), NUMERICOS),
    ])


def montar_modelo(estimador):
    """Pré-processamento + classificador num objeto só."""
    return Pipeline([("preparo", montar_preparo()), ("classificador", estimador)])


# Verificação: o preparo é ajustado só no treino e depois aplicado ao teste.
preparo_teste = montar_preparo().fit(X_treino)
print("colunas após o preparo:", preparo_teste.transform(X_teste).shape[1])

colunas após o preparo: 22


## 5. Baselines comparados na validação

Antes de testar modelos mais complexos, avaliamos três referências nas 4 dobras
de validação temporal (`TimeSeriesSplit`), mantendo o conjunto de teste
intocado:

- **Trivial (`DummyClassifier`):** prevê sempre a classe majoritária. Funciona
  como piso mínimo — qualquer modelo útil precisa superá-lo na capacidade de
  ordenar casos por risco.
- **Regressão Logística:** baseline linear regularizado, rápido e interpretável.
- **Árvore de Decisão (`max_depth=5`):** baseline não linear com profundidade
  limitada para conter sobreajuste.

### 5.1 O que estamos medindo e por que o F1 do trivial engana

A comparação usa **ROC-AUC** (capacidade de atribuir probabilidades maiores a
quem realmente ficará inadimplente) e **F1** (média harmônica entre precisão e
revocação no limiar padrão de 0,50).

**Armadilha na leitura do F1.** Como a classe positiva
(`inadimplente_proximo_exercicio = 1`) representa cerca de 50,8% do treino, um
classificador trivial que chuta sempre `1` obtém revocação de 100% e um F1
aparentemente alto (~0,67), mas tem **ROC-AUC = 0,5000** — equivalente a jogar
uma moeda, sem nenhuma utilidade para priorizar a fila. Por isso o ROC-AUC é o
critério principal de comparação.

In [ ]:
def comparar_na_validacao(candidatos, X, y, cv=validacao_temporal):
    """Calcula ROC-AUC e F1 médios (e desvio-padrão) nas dobras temporais."""
    linhas = []
    for nome, estimador in candidatos.items():
        notas = cross_validate(
            montar_modelo(estimador),
            X,
            y,
            cv=cv,
            scoring=["roc_auc", "f1"],
            n_jobs=-1,
        )
        linhas.append({
            "modelo": nome,
            "roc_auc_media": notas["test_roc_auc"].mean(),
            "roc_auc_dp": notas["test_roc_auc"].std(),
            "f1_media": notas["test_f1"].mean(),
            "f1_dp": notas["test_f1"].std(),
        })
    return pd.DataFrame(linhas).round(4)


baselines = {
    "trivial (maioria)": DummyClassifier(strategy="most_frequent"),
    "regressão logística": LogisticRegression(max_iter=1000, random_state=SEMENTE),
    "árvore de decisão": DecisionTreeClassifier(max_depth=5, random_state=SEMENTE),
}

placar_baselines = comparar_na_validacao(baselines, X_treino, y_treino)
print(placar_baselines.to_string(index=False))

             modelo  roc_auc_media  roc_auc_dp  f1_media  f1_dp
  trivial (maioria)         0.5000      0.0000    0.6718 0.0063
regressão logística         0.7119      0.0116    0.6658 0.0106
  árvore de decisão         0.6884      0.0165    0.6506 0.0238


### 5.2 Síntese da seção 5

- **Piso de referência estabelecido:** o classificador trivial obteve ROC-AUC de
  `0,5000` (F1 de `[inserir F1]`), confirmando que o F1 isolado não distingue um
  chute constante de um modelo útil.
- **Sinal preditivo confirmado:** a Regressão Logística alcançou ROC-AUC médio
  de `[inserir ROC-AUC]` e a Árvore de Decisão `[inserir ROC-AUC]` nas dobras
  temporais, demonstrando que os 12 atributos selecionados carregam informação
  consistente sobre a inadimplência futura.
- **Integridade da validação:** como cada candidato passa por `montar_modelo()`,
  o pré-processamento foi reajustado do zero dentro de cada dobra do
  `TimeSeriesSplit`.

## 6. Modelo com tunagem e escolha do vencedor

Nesta seção avaliamos um comitê de árvores (**Random Forest**) por meio de busca
em grade enxuta (`GridSearchCV`), variando número de árvores (`n_estimators`),
profundidade máxima (`max_depth`) e tamanho mínimo das folhas
(`min_samples_leaf`), otimizando o **ROC-AUC**.

### 6.1 Como a tunagem evita vazamento de dados

O `GridSearchCV` recebe o `Pipeline` completo (`montar_modelo`), e não apenas o
classificador. Assim, a cada combinação de hiperparâmetros e a cada dobra do
`TimeSeriesSplit`, a imputação pela mediana, a padronização (`StandardScaler`) e
a codificação *one-hot* são ajustadas exclusivamente na fração de treino daquela
dobra.

### 6.2 Critério de escolha e variação entre dobras

Após encontrar a melhor configuração do Random Forest, colocamos o modelo tunado
lado a lado com os baselines da seção 5 nas mesmas 4 dobras temporais. Como
mostrado na seção 3.3, oscilações de poucos milésimos no ROC-AUC ficam dentro do
desvio-padrão entre dobras (`roc_auc_dp`); o vencedor é selecionado pelo maior
ROC-AUC médio e reajustado em todo o conjunto `X_treino`.

In [ ]:
grade_rf = {
    "classificador__n_estimators": [150, 300],
    "classificador__max_depth": [6, 10, None],
    "classificador__min_samples_leaf": [5, 20],
}

busca_rf = GridSearchCV(
    estimator=montar_modelo(RandomForestClassifier(random_state=SEMENTE, n_jobs=-1)),
    param_grid=grade_rf,
    cv=validacao_temporal,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True,
)
busca_rf.fit(X_treino, y_treino)

print("Melhores hiperparâmetros (RF):", busca_rf.best_params_)
print(f"Melhor ROC-AUC na validação (RF): {busca_rf.best_score_:.4f}")

# Comparação direta na mesma validação temporal entre todos os candidatos
candidatos_finais = {
    **baselines,
    "random forest (tunado)": clone(busca_rf.best_estimator_.named_steps["classificador"]),
}
placar_geral = comparar_na_validacao(candidatos_finais, X_treino, y_treino)
print("\nPlacar consolidado na validação temporal:")
print(placar_geral.to_string(index=False))

# Seleção automática do vencedor pelo maior ROC-AUC médio na validação
melhor_nome = placar_geral.sort_values("roc_auc_media", ascending=False).iloc[0]["modelo"]
modelo_m1 = montar_modelo(clone(candidatos_finais[melhor_nome])).fit(X_treino, y_treino)
print(f"\nModelo vencedor selecionado para o Modelo 1: {melhor_nome}")

Melhores hiperparâmetros (RF): {'classificador__max_depth': 6, 'classificador__min_samples_leaf': 5, 'classificador__n_estimators': 300}
Melhor ROC-AUC na validação (RF): 0.7068

Placar consolidado na validação temporal:
                modelo  roc_auc_media  roc_auc_dp  f1_media  f1_dp
     trivial (maioria)         0.5000      0.0000    0.6718 0.0063
   regressão logística         0.7119      0.0116    0.6658 0.0106
     árvore de decisão         0.6884      0.0165    0.6506 0.0238
random forest (tunado)         0.7068      0.0132    0.6742 0.0084

Modelo vencedor selecionado para o Modelo 1: regressão logística


### 6.3 Síntese da seção 6

- **Melhor configuração encontrada:** `[inserir best_params_]` com ROC-AUC médio
  de `[inserir best_score_]` na validação temporal.
- **Modelo vencedor (Modelo 1):** `[inserir melhor_nome]` apresentou o maior
  ROC-AUC médio (`[inserir valor] ± [inserir dp]`) e foi reajustado sobre os
  14.877 registros de treino.
- **Conjunto de teste preservado:** nenhuma decisão de hiperparâmetro ou
  seleção de algoritmo utilizou `X_teste`.

## 7. Limiar, avaliação final e métricas por subgrupo

Com o algoritmo vencedor definido, executamos três etapas finais para o
Modelo 1:

1. **Calibração do limiar de decisão:** testamos cortes de probabilidade entre
   0,35 e 0,65 usando apenas previsões fora-da-amostra geradas nas dobras de
   validação de `X_treino`, escolhendo o ponto que maximiza o F1 sem distorcer
   precisão e revocação.
2. **Avaliação única no teste:** aplicamos `modelo_m1` uma única vez sobre
   `X_teste` (3.730 registros, abr/2026 a out/2026) e calculamos Acurácia,
   Precisão, Recall, F1, ROC-AUC e Matriz de Confusão.
3. **Auditoria por subgrupo:** medimos o desempenho separadamente por
   `faixa_renda_setor` e por `setor_urbano` para verificar se o modelo trata as
   regiões e faixas de renda da cidade de forma equilibrada.

In [ ]:
# 1) Escolha do limiar apenas dentro da validação temporal (sem tocar no teste)
probs_val = []
ys_val = []
for idx_tr, idx_va in validacao_temporal.split(X_treino):
    m_fold = montar_modelo(clone(modelo_m1.named_steps["classificador"]))
    m_fold.fit(X_treino.iloc[idx_tr], y_treino.iloc[idx_tr])
    probs_val.append(m_fold.predict_proba(X_treino.iloc[idx_va])[:, 1])
    ys_val.append(y_treino.iloc[idx_va].to_numpy())

probs_val = np.concatenate(probs_val)
ys_val = np.concatenate(ys_val)

limiares = np.arange(0.35, 0.66, 0.05)
tabela_limiares = pd.DataFrame([
    {
        "limiar": round(float(t), 2),
        "precisao": precision_score(ys_val, probs_val >= t, zero_division=0),
        "recall": recall_score(ys_val, probs_val >= t, zero_division=0),
        "f1": f1_score(ys_val, probs_val >= t, zero_division=0),
    }
    for t in limiares
]).round(4)

print("Sensibilidade ao limiar (validação temporal):")
print(tabela_limiares.to_string(index=False))

LIMIAR_M1 = float(tabela_limiares.sort_values("f1", ascending=False).iloc[0]["limiar"])
print(f"\nLimiar escolhido na validação: {LIMIAR_M1:.2f}")

# 2) Avaliação única no conjunto de teste
prob_teste_m1 = modelo_m1.predict_proba(X_teste)[:, 1]
pred_teste_m1 = (prob_teste_m1 >= LIMIAR_M1).astype(int)

metricas_teste_m1 = {
    "roc_auc": round(float(roc_auc_score(y_teste, prob_teste_m1)), 4),
    "acuracia": round(float(accuracy_score(y_teste, pred_teste_m1)), 4),
    "precisao": round(float(precision_score(y_teste, pred_teste_m1, zero_division=0)), 4),
    "recall": round(float(recall_score(y_teste, pred_teste_m1, zero_division=0)), 4),
    "f1": round(float(f1_score(y_teste, pred_teste_m1, zero_division=0)), 4),
}

print("\nMétricas finais no conjunto de teste (Modelo 1):")
for k, v in metricas_teste_m1.items():
    print(f"  {k:10s}: {v:.4f}")

cm = confusion_matrix(y_teste, pred_teste_m1)
df_cm = pd.DataFrame(
    cm,
    index=["Real: Adimplente (0)", "Real: Inadimplente (1)"],
    columns=["Previsto: Adimplente (0)", "Previsto: Inadimplente (1)"],
)
print("\nMatriz de Confusão (teste):")
print(df_cm)

Sensibilidade ao limiar (validação temporal):
 limiar  precisao  recall     f1
   0.35    0.5714  0.8885 0.6955
   0.40    0.5975  0.8274 0.6939
   0.45    0.6217  0.7673 0.6868
   0.50    0.6546  0.6776 0.6659
   0.55    0.6848  0.5854 0.6312
   0.60    0.7107  0.4857 0.5771
   0.65    0.7634  0.3726 0.5008

Limiar escolhido na validação: 0.35

Métricas finais no conjunto de teste (Modelo 1):
  roc_auc   : 0.7069
  acuracia  : 0.6113
  precisao  : 0.5758
  recall    : 0.8658
  f1        : 0.6916

Matriz de Confusão (teste):
                        Previsto: Adimplente (0)  Previsto: Inadimplente (1)
Real: Adimplente (0)                         654                        1198
Real: Inadimplente (1)                       252                        1626


In [ ]:
def avaliar_por_subgrupo(df_base, y_real, y_pred, y_prob, coluna_grupo):
    """Calcula métricas de desempenho e taxa de seleção por subgrupo."""
    analise = pd.DataFrame({
        "grupo": df_base[coluna_grupo].values,
        "y_real": y_real.values,
        "y_pred": y_pred,
        "y_prob": y_prob,
    })
    linhas = []
    for grupo, g in analise.groupby("grupo", observed=False):
        auc = roc_auc_score(g["y_real"], g["y_prob"]) if g["y_real"].nunique() > 1 else np.nan
        linhas.append({
            coluna_grupo: grupo,
            "n": len(g),
            "taxa_real": g["y_real"].mean(),
            "taxa_selecao": g["y_pred"].mean(),
            "precisao": precision_score(g["y_real"], g["y_pred"], zero_division=0),
            "recall": recall_score(g["y_real"], g["y_pred"], zero_division=0),
            "f1": f1_score(g["y_real"], g["y_pred"], zero_division=0),
            "roc_auc": auc,
        })
    res = pd.DataFrame(linhas).round(4)
    dif_selecao = res["taxa_selecao"].max() - res["taxa_selecao"].min()
    dif_recall = res["recall"].max() - res["recall"].min()
    return res, round(float(dif_selecao), 4), round(float(dif_recall), 4)


for col_subgrupo in ["faixa_renda_setor", "setor_urbano"]:
    tabela_sub, dif_sel, dif_rec = avaliar_por_subgrupo(
        teste, y_teste, pred_teste_m1, prob_teste_m1, col_subgrupo
    )
    print(f"\nDesempenho por {col_subgrupo}:")
    print(tabela_sub.to_string(index=False))
    print(f"  Diferença máx. taxa de seleção: {dif_sel:.4f} | Diferença máx. recall: {dif_rec:.4f}")


Desempenho por faixa_renda_setor:
faixa_renda_setor    n  taxa_real  taxa_selecao  precisao  recall     f1  roc_auc
             alta  812     0.3966        0.4495    0.5041  0.5714 0.5357   0.6457
            baixa 1572     0.5999        0.8944    0.6287  0.9374 0.7527   0.6782
            media 1346     0.4554        0.7823    0.5299  0.9103 0.6699   0.7172
  Diferença máx. taxa de seleção: 0.4449 | Diferença máx. recall: 0.3660

Desempenho por setor_urbano:
           setor_urbano   n  taxa_real  taxa_selecao  precisao  recall     f1  roc_auc
       Area de Expansao 185     0.4703        0.8216    0.5263  0.9195 0.6695   0.7314
                 Aureny 613     0.5971        0.8777    0.6283  0.9235 0.7478   0.6828
       Jardim Aureny IV 266     0.4925        0.7669    0.5784  0.9008 0.7045   0.7365
    Plano Diretor Norte 568     0.4049        0.4472    0.4921  0.5435 0.5165   0.6234
      Plano Diretor Sul 895     0.4413        0.7788    0.5165  0.9114 0.6593   0.7094
Setor Comerc

### 7.1 Leitura operacional da matriz de confusão e dos subgrupos

- **Custo dos erros:** na matriz de confusão, **falsos positivos** representam
  contribuintes adimplentes sinalizados como risco (geram custo administrativo
  de comunicação preventiva, sem sanção), enquanto **falsos negativos** são
  inadimplentes futuros não antecipados pelo alerta.
- **Equidade territorial e de renda:** embora `faixa_renda_setor` tenha ficado
  fora dos preditores (`PROIBIDOS`), atributos espaciais como `setor_urbano` e
  `distancia_centro_km` podem carregar correlação indireta com renda. A auditoria
  por subgrupo expõe eventuais diferenças na taxa de seleção e no recall.
- **Papel restrito do Modelo 1:** conforme as diretrizes de governança do
  projeto, a predição de inadimplência futura serve apenas para planejamento
  agregado e campanhas educativas — a fila ativa de cobrança é ordenada pelo
  **Modelo 2** (recuperabilidade).

### 7.2 Síntese da seção 7

- **Limiar operacional calibrado na validação:** `[inserir LIMIAR_M1]`.
- **Desempenho final no teste (uso único):** ROC-AUC = `[inserir]`, Acurácia =
  `[inserir]`, Precisão = `[inserir]`, Recall = `[inserir]` e F1 = `[inserir]`.
- **Auditoria de subgrupos:** diferença máxima de taxa de seleção de
  `[inserir dif_sel]` por faixa de renda e `[inserir dif_sel]` por setor urbano.

## 8. O segundo modelo: scoring de recuperabilidade

Para apoiar a equipe de cobrança da dívida ativa, a pergunta operacional muda:
não interessa prever *quem vai deixar de pagar*, mas estimar **qual a
probabilidade de recuperar em até 12 meses uma dívida já inscrita**
(`divida_recuperada_12m`).

### 8.1 Recorte do universo e ordenação por retorno esperado

- **Filtro de elegibilidade:** o Modelo 2 atua exclusivamente sobre
  contribuintes com dívida ativa inscrita (`valor_divida_consolidada > 0`),
  mantendo exatamente o mesmo corte temporal (`CORTE_TESTE = 2026-04-01`) para
  garantir que o treino ocorra no passado e o teste no futuro.
- **Cálculo do retorno esperado:** combinamos a probabilidade estimada com o
  saldo devedor consolidado:
  $$\text{Retorno esperado} = \text{Valor consolidado} \times P(\text{recuperação em 12m})$$
  Esse critério evita tanto priorizar dívidas muito altas e incobráveis quanto
  gastar esforço apenas em dívidas pequenas e fáceis.
- **Triagem em três faixas operacionais:**
  - **Alta recuperabilidade ($P > 0{,}66$):** cobrança administrativa direta e
    oferta ativa de parcelamento.
  - **Média recuperabilidade ($0{,}33 \le P \le 0{,}66$):** análise caso a caso e
    campanhas segmentadas.
  - **Baixa recuperabilidade ($P < 0{,}33$):** avaliação de custo-benefício
    antes de mobilizar diligência.

In [ ]:
ALVO_M2 = "divida_recuperada_12m"

# Recorte aplicado separadamente em treino e teste (mantendo ordem cronológica)
treino_m2 = treino[treino["valor_divida_consolidada"] > 0].copy()
teste_m2 = teste[teste["valor_divida_consolidada"] > 0].copy()

assert treino_m2["competencia"].is_monotonic_increasing, "treino_m2 fora de ordem!"
assert treino_m2["competencia"].max() < teste_m2["competencia"].min(), "vazamento temporal no M2!"

X_treino_m2, y_treino_m2 = treino_m2[ATRIBUTOS], treino_m2[ALVO_M2]
X_teste_m2, y_teste_m2 = teste_m2[ATRIBUTOS], teste_m2[ALVO_M2]

print(f"Modelo 2 | treino: {len(treino_m2)} | teste: {len(teste_m2)}")

# Comparação na validação temporal para o Modelo 2
candidatos_m2 = {
    "regressão logística": LogisticRegression(max_iter=1000, random_state=SEMENTE),
    "árvore de decisão": DecisionTreeClassifier(max_depth=5, random_state=SEMENTE),
    "random forest": RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=10, random_state=SEMENTE, n_jobs=-1
    ),
}

placar_m2 = comparar_na_validacao(candidatos_m2, X_treino_m2, y_treino_m2)
print("\nPlacar na validação temporal (Modelo 2 — Recuperabilidade):")
print(placar_m2.to_string(index=False))

melhor_nome_m2 = placar_m2.sort_values("roc_auc_media", ascending=False).iloc[0]["modelo"]
modelo_m2 = montar_modelo(clone(candidatos_m2[melhor_nome_m2])).fit(X_treino_m2, y_treino_m2)
print(f"\nModelo vencedor selecionado para o Modelo 2: {melhor_nome_m2}")

# Avaliação única no conjunto de teste do Modelo 2
prob_teste_m2 = modelo_m2.predict_proba(X_teste_m2)[:, 1]
pred_teste_m2 = (prob_teste_m2 >= 0.50).astype(int)

metricas_teste_m2 = {
    "roc_auc": round(float(roc_auc_score(y_teste_m2, prob_teste_m2)), 4),
    "acuracia": round(float(accuracy_score(y_teste_m2, pred_teste_m2)), 4),
    "precisao": round(float(precision_score(y_teste_m2, pred_teste_m2, zero_division=0)), 4),
    "recall": round(float(recall_score(y_teste_m2, pred_teste_m2, zero_division=0)), 4),
    "f1": round(float(f1_score(y_teste_m2, pred_teste_m2, zero_division=0)), 4),
}
print("\nMétricas finais no teste (Modelo 2):")
for k, v in metricas_teste_m2.items():
    print(f"  {k:10s}: {v:.4f}")

Modelo 2 | treino: 12289 | teste: 3072

Placar na validação temporal (Modelo 2 — Recuperabilidade):
             modelo  roc_auc_media  roc_auc_dp  f1_media  f1_dp
regressão logística         0.6717      0.0113    0.6587 0.0115
  árvore de decisão         0.6404      0.0264    0.6412 0.0265
      random forest         0.6601      0.0130    0.6616 0.0173

Modelo vencedor selecionado para o Modelo 2: regressão logística

Métricas finais no teste (Modelo 2):
  roc_auc   : 0.6842
  acuracia  : 0.6367
  precisao  : 0.6423
  recall    : 0.7154
  f1        : 0.6769


In [ ]:
def classificar_faixa_recuperabilidade(prob):
    if prob > 0.66:
        return "Alta (> 66%)"
    elif prob >= 0.33:
        return "Média (33% a 66%)"
    return "Baixa (< 33%)"


fila_teste = teste_m2[[
    "id_contribuinte", "setor_urbano", "faixa_renda_setor",
    "tipo_imovel", "tempo_inadimplencia_dias", "valor_divida_consolidada",
]].copy()
fila_teste["prob_recuperacao"] = prob_teste_m2
fila_teste["retorno_esperado"] = (
    fila_teste["valor_divida_consolidada"] * fila_teste["prob_recuperacao"]
)
fila_teste["faixa_recuperabilidade"] = fila_teste["prob_recuperacao"].apply(
    classificar_faixa_recuperabilidade
)
fila_teste["recuperada_real"] = y_teste_m2.values
fila_teste["valor_recuperado_real"] = (
    fila_teste["valor_divida_consolidada"] * fila_teste["recuperada_real"]
)

# 1) Tabela de calibração das faixas
ordem_faixas = ["Alta (> 66%)", "Média (33% a 66%)", "Baixa (< 33%)"]
calibracao = (
    fila_teste.groupby("faixa_recuperabilidade", observed=False)
    .agg(
        casos=("id_contribuinte", "count"),
        prob_prevista_media=("prob_recuperacao", "mean"),
        taxa_recuperacao_real=("recuperada_real", "mean"),
        divida_total_rs=("valor_divida_consolidada", "sum"),
        retorno_esperado_rs=("retorno_esperado", "sum"),
    )
    .reindex(ordem_faixas)
)
calibracao["pct_carteira"] = (calibracao["casos"] / len(fila_teste) * 100).round(1)
calibracao["prob_prevista_media"] = calibracao["prob_prevista_media"].round(4)
calibracao["taxa_recuperacao_real"] = calibracao["taxa_recuperacao_real"].round(4)
calibracao["divida_total_rs"] = calibracao["divida_total_rs"].round(2)
calibracao["retorno_esperado_rs"] = calibracao["retorno_esperado_rs"].round(2)

print("Tabela de calibração das faixas de recuperabilidade (teste):")
print(calibracao.to_string())

# 2) Curva de concentração ao ordenar por retorno esperado
fila_ordenada = fila_teste.sort_values("retorno_esperado", ascending=False).reset_index(drop=True)
total_recuperado = fila_ordenada["valor_recuperado_real"].sum()

print("\nConcentração do valor recuperado ao priorizar por retorno esperado:")
for pct in [0.10, 0.20, 0.30, 0.50]:
    topo = int(len(fila_ordenada) * pct)
    fatia_rec = fila_ordenada.iloc[:topo]["valor_recuperado_real"].sum() / total_recuperado * 100
    print(f"  Top {int(pct*100):2d}% da fila ({topo:4d} casos) -> {fatia_rec:5.1f}% do valor recuperado")

Tabela de calibração das faixas de recuperabilidade (teste):
                        casos  prob_prevista_media  taxa_recuperacao_real  divida_total_rs  retorno_esperado_rs  pct_carteira
faixa_recuperabilidade                                                                                                       
Alta (> 66%)              541               0.7089                 0.7301       1432590.94           1014530.77          17.6
Média (33% a 66%)        2081               0.5220                 0.5464       8123470.11           4119030.65          67.7
Baixa (< 33%)             450               0.2189                 0.2267       2978552.06            608743.68          14.6

Concentração do valor recuperado ao priorizar por retorno esperado:
  Top 10% da fila ( 307 casos) ->  36.2% do valor recuperado
  Top 20% da fila ( 614 casos) ->  52.8% do valor recuperado
  Top 30% da fila ( 921 casos) ->  64.1% do valor recuperado
  Top 50% da fila (1536 casos) ->  80.2% do valor recuper

### 8.2 Síntese da seção 8

- **Universo e vencedor (Modelo 2):** treinado sobre `[inserir len(treino_m2)]`
  registros com dívida inscrita e avaliado em `[inserir len(teste_m2)]` casos de
  teste, com vencedor `[inserir melhor_nome_m2]` (ROC-AUC de teste = `[inserir]`).
- **Calibração das faixas:** as probabilidades médias previstas nas faixas Alta,
  Média e Baixa acompanham as taxas reais observadas no teste (`[inserir taxas
  reais]`), validando o uso das faixas como triagem assistida.
- **Efeito de concentração:** ao ordenar a fila de teste por `retorno_esperado`,
  os **20% melhores casos** concentram `[inserir %]%` de todo o valor
  efetivamente recuperado, e a **metade superior da carteira** alcança
  `[inserir %]%`.

## 9. Serialização e versionamento

Para que o painel interativo em Streamlit (entregável E4) execute previsões reais
sem precisar retreinar os modelos, exportamos os dois objetos `Pipeline`
completos (`pré-processamento + classificador`) para a pasta `/models` em
formato `.joblib`, acompanhados de um manifesto JSON (`metadados_modelos.json`).

### 9.1 Garantias de reprodutibilidade e teste de recarga

- **Pipeline autocontido:** salvar o `Pipeline` inteiro garante que qualquer
  novo registro recebido pelo painel passe exatamente pela mesma imputação,
  escala e codificação ajustadas no treino.
- **Rastreabilidade:** o arquivo `metadados_modelos.json` registra a semente
  (`SEMENTE = 42`), as versões do Python e do `scikit-learn`, a lista exata de
  atributos de entrada, o limiar calibrado e a regra obrigatória de exclusão do
  `iptu_social`.
- **Verificação pós-exportação (*smoke test*):** recarregamos ambos os arquivos
  `.joblib` do disco e confirmamos via `np.allclose` que as probabilidades
  previstas são idênticas às calculadas em memória.

In [ ]:
caminho_m1 = MODELOS / "modelo_inadimplencia.joblib"
caminho_m2 = MODELOS / "modelo_recuperabilidade.joblib"
caminho_meta = MODELOS / "metadados_modelos.json"

joblib.dump(modelo_m1, caminho_m1)
joblib.dump(modelo_m2, caminho_m2)

metadados = {
    "data_treinamento": datetime.now().isoformat(timespec="seconds"),
    "semente": SEMENTE,
    "versoes": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "atributos_entrada": ATRIBUTOS,
    "categoricos": CATEGORICOS,
    "numericos": NUMERICOS,
    "exclusao_obrigatoria": "iptu_social == 0",
    "modelo_1_inadimplencia": {
        "arquivo": caminho_m1.name,
        "algoritmo": melhor_nome,
        "alvo": ALVO,
        "limiar_operacional": LIMIAR_M1,
        "metricas_teste": metricas_teste_m1,
    },
    "modelo_2_recuperabilidade": {
        "arquivo": caminho_m2.name,
        "algoritmo": melhor_nome_m2,
        "alvo": ALVO_M2,
        "filtro_universo": "valor_divida_consolidada > 0",
        "faixas": {"alta": "> 0.66", "media": "0.33 a 0.66", "baixa": "< 0.33"},
        "metricas_teste": metricas_teste_m2,
    },
}

caminho_meta.write_text(json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")

# Teste de recarga (smoke test) para garantir que o painel Streamlit carregará sem erros
m1_recarregado = joblib.load(caminho_m1)
m2_recarregado = joblib.load(caminho_m2)

assert np.allclose(
    m1_recarregado.predict_proba(X_teste.iloc[:5])[:, 1],
    prob_teste_m1[:5],
), "Divergência ao recarregar modelo_inadimplencia.joblib!"

assert np.allclose(
    m2_recarregado.predict_proba(X_teste_m2.iloc[:5])[:, 1],
    prob_teste_m2[:5],
), "Divergência ao recarregar modelo_recuperabilidade.joblib!"

print("Arquivos salvos e verificados em /models:")
for arq in sorted(MODELOS.iterdir()):
    print(f"  - {arq.name} ({arq.stat().st_size / 1024:.1f} KB)")

Arquivos salvos e verificados em /models:
  - metadados_modelos.json (1.7 KB)
  - modelo_inadimplencia.joblib (5.1 KB)
  - modelo_recuperabilidade.joblib (5.1 KB)


### 9.2 Síntese da seção 9

- **Artefatos versionados em `/models`:** `modelo_inadimplencia.joblib`,
  `modelo_recuperabilidade.joblib` e `metadados_modelos.json` salvos e
  validados sem divergência numérica na recarga.
- **Trilha concluída:** o notebook executa de ponta a ponta (carga → exclusão
  do grupo protegido → split temporal → pipeline → baselines → tunagem →
  avaliação única → scoring → serialização) com caminhos relativos e semente
  fixa.